In [1]:
import boto3
import pandas as pd

In [7]:
#creamos el cliente y el bucket
execution_role = (
    "arn:aws:iam::066401718601:role/"
    "service-role/AmazonSageMaker-ExecutionRole-20260820T174073"
)
s3 = boto3.client("s3")
bucket = "telco-constumer-churn-066401718601-us-east-2-an"

In [9]:
#obtener el csv de validacion
response = s3.get_object(
    Bucket=bucket,
    Key="Processed/val.csv"
)

test = pd.read_csv(response["Body"])

y_test = test["Churn"]
X_test = test.drop(columns=["Churn"])

print(X_test.shape)
print(X_test.head())

(1125, 45)
   num__SeniorCitizen  num__tenure  num__MonthlyCharges  num__TotalCharges  \
0           -0.444289     1.587159             0.216567           1.215488   
1           -0.444289    -1.264821            -1.448448          -0.987329   
2           -0.444289    -1.264821            -1.461742          -0.987507   
3           -0.444289    -1.224079             0.377751          -0.912406   
4           -0.444289    -1.264821            -1.476697          -0.987707   

   cat__Partner_No  cat__Partner_Yes  cat__Dependents_No  cat__Dependents_Yes  \
0              0.0               1.0                 0.0                  1.0   
1              1.0               0.0                 0.0                  1.0   
2              1.0               0.0                 1.0                  0.0   
3              1.0               0.0                 1.0                  0.0   
4              1.0               0.0                 1.0                  0.0   

   cat__PhoneService_No  cat__Pho

In [10]:
#tomar una observacion
sample = X_test.iloc[[0]]

print(sample)

   num__SeniorCitizen  num__tenure  num__MonthlyCharges  num__TotalCharges  \
0           -0.444289     1.587159             0.216567           1.215488   

   cat__Partner_No  cat__Partner_Yes  cat__Dependents_No  cat__Dependents_Yes  \
0              0.0               1.0                 0.0                  1.0   

   cat__PhoneService_No  cat__PhoneService_Yes  ...  cat__Contract_One year  \
0                   0.0                    1.0  ...                     0.0   

   cat__Contract_Two year  cat__PaperlessBilling_No  \
0                     1.0                       0.0   

   cat__PaperlessBilling_Yes  cat__PaymentMethod_Bank transfer (automatic)  \
0                        1.0                                           0.0   

   cat__PaymentMethod_Credit card (automatic)  \
0                                         1.0   

   cat__PaymentMethod_Electronic check  cat__PaymentMethod_Mailed check  \
0                                  0.0                              0.0   

   

In [12]:
#convertimos dicha observacion en csv
payload = sample.to_csv(
    header=False,
    index=False
)

print(payload)

-0.4442892342268793,1.5871594309348782,0.2165666719613066,1.215487834611754,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0



In [14]:

runtime = boto3.client("sagemaker-runtime", region_name="us-east-2")

In [15]:
#nombramos el endpoint
endpoint_name = "telco-churn-xgboost-endpoint"

In [18]:
#invocamos el endpoint y realizamos la prueba
response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="text/csv",
    Body=payload
)

result = response["Body"].read().decode("utf-8")

print("Respuesta del endpoint:")
print(result)

Respuesta del endpoint:
0.009815942496061325



In [19]:
#convertimos dicha respuesta a 1 o 0
probability = float(result)

prediction = int(probability >= 0.5)

print(f"Probabilidad de Churn: {probability:.4f}")
print(f"Predicción: {prediction}")

Probabilidad de Churn: 0.0098
Predicción: 0


In [20]:
#apagar el endpoint

sm = boto3.client("sagemaker", region_name="us-east-2")

sm.delete_endpoint(
    EndpointName="telco-churn-xgboost-endpoint"
)

print("✅ Endpoint eliminado")

✅ Endpoint eliminado


In [21]:
response = sm.describe_endpoint(
    EndpointName="telco-churn-xgboost-endpoint"
)

print(response["EndpointStatus"])

ClientError: An error occurred (ValidationException) when calling the DescribeEndpoint operation: Could not find endpoint "telco-churn-xgboost-endpoint".